<a href="https://colab.research.google.com/github/EstevaoMO/machine-learning-aprendizadodemaquina/blob/main/01_fundamentos_de_ml/colabs/Fundamentos_de_Machine_Learning_Entendendo_Estruturas_e_Dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fundamentos de Machine Learning: Entendendo Estruturas e Dados**

In [6]:
# %pip install pandas seaborn scikit-learn

## **1. A Lógica da Inferência**

*Não se preocupe em decorar tudo que está aqui, ou entender todas as bibliotecas. Elas são importantes, mas o foco aqui é o desenvolvimento do caráter analítico e entendimento do conteúdo abordado.*

**O que é "Aprender"?**

Na computação tradicional, escrevemos regras explícitas para transformar dados de entrada em saídas. Em Aprendizado de Máquina (ML), invertemos essa relação: fornecemos as entradas e as saídas esperadas para que o algoritmo infira as regras matemáticas subjacentes.

Segundo a definição formal de Tom Mitchell (1997):

>"Um programa de computador aprende a partir da experiência $E$, com relação a uma classe de tarefas $T$ e medida de desempenho $P$, se o seu desempenho em tarefas em $T$, medido por $P$, melhora com a experiência $E$."

### A Transição do Pensamento Analítico
Para ilustrar, usaremos o famoso manifesto de passageiros do RMS Titanic.

- **Análise Descritiva e Prescritiva:** Olham para o passado ($E$). Focam em responder: "Quantas pessoas sobreviveram?" ou "Qual era a média de idade da 3ª classe?".
- **Análise Preditiva (Inferência Estatística):** Utiliza o passado para projetar cenários não observados. Pergunta-se: "Dado um novo passageiro com características específicas, qual a probabilidade matemática de sua sobrevivência?".

**O modelo atua por indução: ele observa uma amostra finita de eventos (os passageiros conhecidos) e generaliza uma função oculta $f$ que rege a probabilidade de sobrevivência.**

In [7]:
import pandas as pd
import seaborn as sns

# Carregando o dataset real do Titanic
titanic = sns.load_dataset('titanic')

# Visualizando as primeiras observações
titanic[['survived', 'pclass', 'sex', 'age', 'fare']].head()

,survived,pclass,sex,age,fare
0,0,3,male,22.0,7.2500
1,1,1,female,38.0,71.2833
2,1,3,female,26.0,7.9250
3,1,1,female,35.0,53.1000
4,0,3,male,35.0,8.0500


**Observe as colunas acima.**

O desafio do Aprendizado de Máquina é responder: quais destas variáveis (classe, idade, sexo) possuem capacidade explicativa suficiente para antecipar quem sobreviveu à tragédia?

## **2. A Anatomia dos Dados Matemáticos ($X$ e $y$)**

### A Exigência Vetorial
Algoritmos matemáticos não interpretam conceitos humanos como "masculino", "feminino" ou "primeira classe". Eles operam exclusivamente sobre matrizes e vetores numéricos.

No aprendizado supervisionado, dividimos nossa estrutura de dados em dois componentes fundamentais:

- **Matriz de Features ($X \in \mathbb{R}^{n \times p}$):** Contém $n$ observações e $p$ variáveis explicativas (preditores). É o contexto do problema.
- **Vetor-Alvo ($y \in \mathbb{R}^n$):** Contém o "gabarito" ou resposta real para cada uma das $n$ observações.

**O objetivo do modelo supervisionado é encontrar uma função matemática $f$ tal que:**

$$y \approx f(X) + \epsilon$$

Onde $\epsilon$ representa o erro irredutível (ruído inerente ao mundo real).

In [8]:
# Preparando os dados: removendo nulos e convertendo texto em números (ex: sexo)
dados_limpos = titanic[['survived', 'pclass', 'sex', 'age', 'fare']].dropna()
dados_limpos['sex_code'] = (dados_limpos['sex'] == 'female').astype(int) # 1 para mulher, 0 para homem

# Isolando a Matriz de Features (X) e o Vetor-Alvo (y)
X = dados_limpos[['pclass', 'sex_code', 'age', 'fare']]
y = dados_limpos['survived']

print(f"Dimensão da Matriz X: {X.shape} -> ({X.shape[0]} passageiros, {X.shape[1]} variáveis)")
print(f"Dimensão do Vetor y: {y.shape} -> ({y.shape[0]} rótulos de sobrevivência)")

Dimensão da Matriz X: (714, 4) -> (714 passageiros, 4 variáveis)
Dimensão do Vetor y: (714,) -> (714 rótulos de sobrevivência)


## **3. Paradigmas de Predição Supervisionada (Regressão vs. Classificação)**

Dependendo da natureza do vetor $y$, o problema de aprendizado supervisionado se divide em dois grandes domínios. Podemos explorar ambos usando o próprio Titanic:

| Critério | Regressão | Classificação |
| --- | --- | --- |
| **Natureza de $y$** | Contínua ($y \in \mathbb{R}$) | Discreta/Categórica ($y \in \{C_1, C_2, \dots, C_k\}$) |
| **Exemplo no Titanic** | Prever o **valor da tarifa (fare)** pago por um passageiro | Prever a **sobrevivência (0 ou 1)** do passageiro |
| **Saída do Modelo** | Um valor escalar numérico (ex: $ 35.50) | Um vetor de probabilidades para a classe |

### Probabilidades e Limiares de Decisão (*Decision Thresholds*)

Na classificação, a maioria dos modelos não devolve a resposta final de forma direta. Em vez disso, calculam a probabilidade contínua de pertencimento a uma classe: $P(Y = 1 \mid X)$.

Para converter essa probabilidade em uma decisão categórica, aplica-se um **limiar de decisão** ($\tau$):

$$\text{Classe} = \begin{cases} 1 & \text{se } P(Y = 1 \mid X) \ge \tau \\ 0 & \text{se } P(Y = 1 \mid X) < \tau \end{cases}$$

In [9]:
from sklearn.linear_model import LogisticRegression

# Treinando um classificador probabilístico
modelo_classificacao = LogisticRegression()
modelo_classificacao.fit(X, y)

# Previsão para os 3 primeiros passageiros
probabilidades = modelo_classificacao.predict_proba(X[:3])[:, 1] # Probabilidade de y=1 (Sobreviver)

print("--- CLASSIFICAÇÃO: PROBABILIDADE VS DECISÃO ---")
print(f"Probabilidades de Sobreviver: {probabilidades.round(3)}")

# Aplicando diferentes limiares (Thresholds)
print(f"Decisão (Corte conservador 0.50): {(probabilidades >= 0.50).astype(int)}")
print(f"Decisão (Corte estrito 0.75):     {(probabilidades >= 0.75).astype(int)}")

--- CLASSIFICAÇÃO: PROBABILIDADE VS DECISÃO ---
Probabilidades de Sobreviver: [0.112 0.905 0.548]
Decisão (Corte conservador 0.50): [0 1 1]
Decisão (Corte estrito 0.75):     [0 1 0]


## **4. O Desafio da Generalização (Underfitting vs. Overfitting)**

O objetivo final do Aprendizado de Máquina é manter alto desempenho em **dados inéditos** (capacidade de generalização).

### Os Dois Extremos da Falha de Aprendizado

- **Underfitting (Alto Viés):** O modelo é simples demais. Exemplo: Um modelo que diz que *ninguém* sobreviveu. Ele tem alto erro no treino e no teste.
- **Overfitting (Alta Variância):** O modelo se ajusta excessivamente ao ruído. Exemplo: Um algoritmo de árvore que cria uma regra dizendo *"Se o passageiro tem 22 anos, é homem, pagou $7.25 e embarcou em Southampton, ele morre"*.

O modelo "decorou" o passageiro *Mr. Owen Harris Braund*, mas essa regra ultraespecífica é inútil para prever sobreviventes em um outro naufrágio.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Separação entre dados de Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Modelo 1: Árvore memorizadora (Overfitting)
modelo_overfit = DecisionTreeClassifier(max_depth=None) # Sem limite de crescimento
modelo_overfit.fit(X_train, y_train)

# Modelo 2: Árvore generalista
modelo_generalista = DecisionTreeClassifier(max_depth=3) # Crescimento restrito
modelo_generalista.fit(X_train, y_train)

print(f"Modelo Memorizador -> Acurácia Treino: {modelo_overfit.score(X_train, y_train):.2f} | Acurácia Teste: {modelo_overfit.score(X_test, y_test):.2f}")
print(f"Modelo Generalista -> Acurácia Treino: {modelo_generalista.score(X_train, y_train):.2f} | Acurácia Teste: {modelo_generalista.score(X_test, y_test):.2f}")


Modelo Memorizador -> Acurácia Treino: 0.99 | Acurácia Teste: 0.74
Modelo Generalista -> Acurácia Treino: 0.83 | Acurácia Teste: 0.76


**Note como o modelo memorizador tem desempenho quase perfeito no passado (treino), mas seu poder preditivo cai drasticamente no futuro (teste).**



## **5. A Natureza Exploratória Sem Gabarito (Não Supervisionado)**

### Quando Não Temos a Resposta ($y$)

O Aprendizado Não Supervisionado opera na ausência absoluta do vetor-alvo $y$. Em vez de prever um resultado, o objetivo é **inferir a estrutura oculta, densidade ou distribuição intrínseca dos dados**.

A tarefa mais comum é o **Agrupamento (Clustering)**: encontrar subgrupos (clusters) onde a distância matemática entre passageiros do mesmo grupo seja mínima, e a distância entre grupos diferentes seja máxima.

### Aplicação no Titanic: Perfis Sociais Ocultos

Imagine que não sabemos quem sobreviveu, nem a classe social do bilhete. Queremos que o algoritmo encontre divisões naturais baseadas apenas na Idade e na Tarifa Paga.

In [11]:
from sklearn.cluster import KMeans

# Isolando apenas Idade e Tarifa (sem rótulos de classe ou sobrevivência)
X_unsupervised = dados_limpos[['age', 'fare']]

# Aplicando agrupamento para descobrir 3 "perfis" ocultos
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
dados_limpos['cluster_social'] = kmeans.fit_predict(X_unsupervised)

# Exibindo os centroides (o "passageiro médio" de cada grupo descoberto)
centroides = pd.DataFrame(kmeans.cluster_centers_, columns=['Idade Média', 'Tarifa Média'])
centroides.index.name = 'Cluster'
centroides.round(2)

,Idade Média,Tarifa Média
Cluster,,
0,31.17,285.38
1,28.36,16.32
2,35.92,85.27


## **6. Conclusão**

### O Que Aprendemos Até Aqui?

Neste primeiro contato, deixamos para trás a postura passiva de apenas "contar o que aconteceu" e assumimos o papel da **inferência preditiva**. Vimos que o Aprendizado de Máquina não é mágica, mas sim a busca matemática por uma função $f$ que consiga mapear um conjunto de entradas ($X$) em uma saída desejada ($y$).

Para consolidar o raciocínio, guarde estes cinco pilares fundamentais:

1. Em vez de programarmos as regras para prever a sobrevivência, nós fornecemos os dados do Titanic (experiência) e deixamos o algoritmo inferir as regras por indução.
2. Para a máquina, não existem "mulheres" ou "primeira classe". Existe apenas a matriz $X$ e o vetor $y$. Todo o seu conhecimento de negócio precisará ser traduzido em números antes de qualquer predição.
3. Se o alvo ($y$) é contínuo (ex: preço da tarifa), fazemos **Regressão**. Se é categórico (ex: sobreviveu ou não), fazemos **Classificação** (apoiada por probabilidades e limiares de decisão matemáticos).
4. Um modelo com $100\%$ de acerto no treino quase sempre está sofrendo de **Overfitting**. Nosso objetivo supremo é a **Generalização**: o modelo precisa performar bem em dados que ele nunca viu.
5. Quando removemos o "gabarito" ($y$), entramos no campo **Não Supervisionado**, forçando o algoritmo a agrupar passageiros puramente pela geometria de suas características (distância matemática).

### Próximos Passos

Note que, ao longo deste notebook, nós ignoramos silenciosamente passageiros com idades faltando ou dados em branco (`dropna()`). No mundo real, dados vêm sujos, incompletos e em formatos que os algoritmos rejeitam (textos soltos, escalas gigantescas misturadas com frações).

Como algoritmos só entendem matrizes limpas, o limite da inteligência do seu modelo será exatamente a qualidade da matriz $X$ que você entregar a ele. Esse conceito é chamado de: **GARBAGE IN. GARBAGE OUT.**

**No próximo material, desceremos para a engenharia de dados (pré-processamento). Aprenderemos como tratar valores ausentes, codificar categorias de texto em matrizes matemáticas e colocar todas as variáveis na mesma escala para que o algoritmo não seja enviesado.**